In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import tqdm
import sys

# Configuration
input_file = "NASI copy.csv"  # Replace with your file name
output_prefix = "out_similarity"  # Prefix for output files
batch_size = 1000  # Number of rows per batch
start_row = 1  # Start processing from this row index

# Load data
df = pd.read_csv(input_file)
all_rows = list(df.iterrows())
all_rows = all_rows[start_row:]  # Start from the specified row index

# Load pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Process data in batches
for batch_idx in range((len(all_rows) // batch_size) + 1):
    print(f"Processing batch {batch_idx + 1}...")
    results = []

    # Get the rows for the current batch
    batch_data = all_rows[batch_size * batch_idx: batch_size * (batch_idx + 1)]

    for i, row in tqdm.tqdm(batch_data):
        try:
            # Extract texts for comparison
            text1 = str(row["reason"]) if pd.notna(row["reason"]) else ""
            text2 = str(row["orig_reason"]) if pd.notna(row["orig_reason"]) else ""

            # Compute embeddings
            embedding1 = model.encode(text1, convert_to_tensor=True)
            embedding2 = model.encode(text2, convert_to_tensor=True)

            # Calculate cosine similarity
            similarity = util.cos_sim(embedding1, embedding2).item()

            # Append the result
            results.append({
                "index": i,
                "reason": text1,
                "orig_reason": text2,
                "Similarity Score": similarity
            })
        except KeyboardInterrupt:
            print("KeyboardInterrupt received, exiting gracefully...")
            sys.exit(0)
        except Exception as e:
            print(f"Failed processing row {i} in batch {batch_idx} due to error: {e}")

    # Save batch results to a CSV file
    if results:
        res_df = pd.DataFrame(results)
        output_file = f"{output_prefix}-{batch_size}-{batch_idx}.csv"
        res_df.to_csv(output_file, index=False)
        print(f"Batch {batch_idx + 1} saved to {output_file}")

print("Batch processing complete.")



Processing batch 1...


100%|██████████| 1000/1000 [01:24<00:00, 11.78it/s]


Batch 1 saved to out_similarity-1000-0.csv
Processing batch 2...


100%|██████████| 1000/1000 [01:17<00:00, 12.85it/s]


Batch 2 saved to out_similarity-1000-1.csv
Processing batch 3...


...

Batch 3 saved to out_similarity-1000-2.csv
Processing batch 4...


100%|██████████| 1000/1000 [01:22<00:00, 12.16it/s]


Batch 4 saved to out_similarity-1000-3.csv
Processing batch 5...


100%|██████████| 141/141 [00:11<00:00, 12.23it/s]

Batch 5 saved to out_similarity-1000-4.csv
Batch processing complete.


 Exact Matches

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('NASI copy.csv')

# Exact matches
exact_matches = (df['reason'] == df['orig_reason']).sum()
exact_match_rate = (exact_matches / len(df)) * 100
print(f"Exact Match Rate: {exact_match_rate:.2f}%")


Exact Match Rate: 0.00%


BLEU Score

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# BLEU scores
bleu_scores = [sentence_bleu([ref.split()], pred.split()) for ref, pred in zip(df['orig_reason'], df['reason'])]
average_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU Score: {average_bleu:.2f}")


/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.11/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Average BLEU Score: 0.00


ROUGE Score

In [ ]:
pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=d3ed2e463a0d1b52a575572cd7f33dab1aec4dd2b4e6a2d03d9d48f9faefecb9
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(ref, pred) for ref, pred in zip(df['orig_reason'], df['reason'])]

# Average ROUGE scores
rouge1_avg = sum(score['rouge1'].fmeasure for score in rouge_scores) / len(rouge_scores)
rougeL_avg = sum(score['rougeL'].fmeasure for score in rouge_scores) / len(rouge_scores)
print(f"Average ROUGE-1 Score: {rouge1_avg:.2f}")
print(f"Average ROUGE-L Score: {rougeL_avg:.2f}")


Average ROUGE-1 Score: 0.12
Average ROUGE-L Score: 0.10


Semantic Similarity

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained BERT model for sentence embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
reason_embeddings = model.encode(df['reason'].tolist())
orig_reason_embeddings = model.encode(df['orig_reason'].tolist())

# Calculate cosine similarity
cosine_similarities = [cosine_similarity([emb1], [emb2])[0][0] for emb1, emb2 in zip(reason_embeddings, orig_reason_embeddings)]
average_similarity = sum(cosine_similarities) / len(cosine_similarities)
print(f"Average Semantic Similarity: {average_similarity:.2f}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Average Semantic Similarity: 0.37
